In [ ]:

import pandas as pd
import datetime
from stock.tw_stock import tw_stock

from talib import SMA, ADX
"""""
雙均線 + ADX 指標策略回測

進場：

快線 SMA(10) > 慢線 SMA(30)

ADX(14) > 25

出場：均線慢線大於快線
"""""


# 一、單支股票日線策略
def 雙均線ADX策略(fast_ma, slow_ma, adx_val, close, adx_threshold=25, x=-1):
    cond1 = fast_ma.iloc[x] > slow_ma.iloc[x]
    cond2 = adx_val.iloc[x] > adx_threshold
    match = cond1 & cond2

    date = close.iloc[x].name
    stocks = close.iloc[x][match].index.values

    return {
        "string": f"{date} 雙均線ADX策略進場：{len(stocks)} 筆股票",
        "stocks": stocks,
        "date": date
    }


In [ ]:
# 二、回測主程式
from tqdm import tqdm
import numpy as np

tw = tw_stock()
start = datetime.datetime(2015, 1, 1)
end = datetime.datetime(2022, 3, 1)

close = tw.get("close", start, end)
high = tw.get("high", start, end)
low = tw.get("low", start, end)

fast_period = 10
slow_period = 30
adx_period = 14
adx_threshold = 25

# 三、預先計算 MA 與 ADX（每支股票個別處理）
fast_ma = pd.DataFrame(index=close.index, columns=close.columns)
slow_ma = pd.DataFrame(index=close.index, columns=close.columns)
adx_val = pd.DataFrame(index=close.index, columns=close.columns)

for stock in close.columns:
    try:
        fast_ma[stock] = SMA(close[stock].values, timeperiod=fast_period)
        slow_ma[stock] = SMA(close[stock].values, timeperiod=slow_period)
        adx_val[stock] = ADX(high[stock].values, low[stock].values, close[stock].values, timeperiod=adx_period)
    except Exception as e:
        print(f"{stock} talib 計算錯誤: {e}")

result = []
position_dict = {}  # key: stock_id, value: (entry_day, entry_price, entry_date)

for i in tqdm(range(100, len(close))):
    try:
        # === 進場 ===
        res = 雙均線ADX策略(fast_ma, slow_ma, adx_val, close, adx_threshold, x=i)
        for stock in res["stocks"]:
            if stock in position_dict:
                continue
            buy_price = close.iloc[i][stock]
            if np.isnan(buy_price):
                continue
            position_dict[stock] = (i, buy_price, res["date"])

        # === 出場 ===
        for stock in list(position_dict.keys()):
            if slow_ma.iloc[i][stock] > fast_ma.iloc[i][stock]:
                i_entry, buy_price, entry_date = position_dict.pop(stock)
                sell_price = close.iloc[i][stock]
                if np.isnan(sell_price):
                    continue
                pct = (sell_price - buy_price) / buy_price
                result.append({
                    "策略": "雙均線+ADX",
                    "股票": stock,
                    "進場日": entry_date,
                    "出場日": close.index[i],
                    "進場價": round(buy_price, 2),
                    "出場價": round(sell_price, 2),
                    "報酬率": round(pct * 100, 2),
                    "持有天數": (close.index[i] - entry_date).days,
                    "結果": "勝利" if pct > 0 else "失敗"
                })

    except Exception as e:
        print(f"第{i}天策略錯誤：{e}")
        continue

# 四、結果數據
result_df = pd.DataFrame(result)
result_df.to_csv("ma_adx_exit_by_ma_cross.csv", index=False)
result_df.head()


In [ ]:
result_df
